<a href="https://colab.research.google.com/github/victorbaraunaAcad/crise-saude-ufam/blob/main/notebooks/01_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — EDA e ampliação da base

**Quando a água some ou sobra demais, quem adoece primeiro?**

Seca no Amazonas (2023–2024) × enchente no Rio Grande do Sul (2024), município a município.

## 1. Ponto de partida

Escrito **antes** de olhar os dados, como pede o enunciado. A Big Idea é provisória e será reescrita depois dos modelos.

| Item | Versão provisória |
|---|---|
| **Pergunta norteadora** | É possível prever se um município terá um mês com internações por doenças ligadas à água acima do seu normal, a partir do tipo de crise (seca ou enchente), do tempo desde o início do evento e do perfil socioeconômico do município? |
| **Persona** | Um jovem de 20 e poucos anos, não técnico, que acompanhou a seca do Rio Negro e a enchente do RS pelo jornal e pelas redes, mas nunca pensou no que acontece com a saúde depois que a notícia sai do ar. |
| **Big Idea (provisória)** | Seca e enchente adoecem de jeitos diferentes, e os municípios mais pobres sentem primeiro e por mais tempo. |
| **ABT** | **E** o Brasil viveu os dois extremos da água em sequência: o Rio Negro no menor nível da história e o RS debaixo d'água. **MAS** a manchete acaba em semanas e a doença não, e ela não atinge todos os municípios igualmente. **PORTANTO** medimos quem adoece, de quê e quando, para mostrar onde a ajuda tem que chegar primeiro. |
| **Alvo (classificação)** | `excesso_drsai`: 1 se a taxa de internações por DRSAI no município/mês ficou acima do normal daquele município. |
| **Alvo (regressão)** | `excesso_taxa_100k`: quantas internações a mais por 100 mil habitantes em relação ao esperado. |

**Hipóteses registradas antes de ver os dados** (para conferir depois se os dados confirmam ou contradizem):

1. A seca é mais brutal no longo prazo, mas a enchente traz um pico forte de doenças (leptospirose, diarreia).
2. Crianças pequenas (0–4 anos) e idosos (60+) são os mais atingidos.
3. Municípios mais pobres são mais afetados; os ricos também sofrem, mas com consequências menores.
4. Possível surpresa: na seca as internações podem **cair**, porque sem rio não há barco e a pessoa não chega ao hospital.

**DRSAI** = Doenças Relacionadas ao Saneamento Ambiental Inadequado (Cairncross e Feachem, 1993; usada com o SIH no RS por Siqueira et al., *Epidemiol. Serv. Saúde*, 2017). Neste projeto: diarreias e infecções intestinais (A00–A09), hepatite A (B15), leptospirose (A27) e dengue (A90, A91, A97). As respiratórias (J00–J22) são analisadas como grupo separado, por causa da fumaça das queimadas na seca.

## 2. Ampliação da base

**Em andamento.** Duas ampliações planejadas (detalhes no `relatorio.md` e em `docs/`):

1. **Municípios atingidos:** Atlas Digital de Desastres / S2iD (Defesa Civil), com tipo e data de cada seca ou enchente reconhecida no AM e no RS, 2022–2024.
2. **Anos de referência:** SIH do RS 2022–2023 e do AM 2022, e SINAN leptospirose 2022–2023, para definir o "normal" de cada município.

Enquanto elas não chegam, esta primeira parte trabalha só com o SIH já coletado (AM 2023–2024, RS 2024). As janelas de seca e enchente nos gráficos abaixo são **aproximadas** e serão trocadas pelas datas oficiais da Ampliação 1.

## Preparação do ambiente

A célula abaixo funciona no Colab e no computador. No Colab, ela baixa o repositório (com os dados do Git LFS) e entra na pasta raiz; daí em diante todos os caminhos são relativos (`data/raw/...`), como o enunciado exige.

In [ ]:
import os, pathlib, subprocess

REPO_URL = "https://github.com/victorbaraunaAcad/crise-saude-ufam.git"

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

if pathlib.Path("data/raw").exists():
    pass                                     # já estamos na raiz do repositório
elif pathlib.Path("../data/raw").exists():
    os.chdir("..")                           # notebook aberto de dentro de notebooks/
else:                                        # Colab: baixa o repositório
    sh("git lfs install || (apt-get -qq install git-lfs && git lfs install)")
    sh(f"GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 {REPO_URL}")
    os.chdir("crise-saude-ufam")

# baixa do LFS só os arquivos que esta parte usa (as internações do AM e do RS)
sh('git lfs pull --include="data/raw/sih_am_2023_2024.parquet,data/raw/sih_rs_2024.parquet"')
print("Pasta de trabalho:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import pyarrow.parquet as pq

SEMENTE = 42
np.random.seed(SEMENTE)
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 40)

RAW = pathlib.Path("data/raw")
PROCESSED = pathlib.Path("data/processed")
FIGURES = pathlib.Path("figures")
FIGURES.mkdir(exist_ok=True)

CORES_UF = {"AM": "#C8742B", "RS": "#2B6CB0"}   # AM = seca (terra), RS = enchente (água)

## 3. Visão geral da base

Carregamos as internações do SIH (Sistema de Informações Hospitalares do SUS). Cada linha é **uma internação** (uma AIH). Lemos só as colunas que a análise usa, para economizar memória:

| Coluna | Significado |
|---|---|
| `N_AIH` | Número da autorização de internação (identificador) |
| `IDENT` | Tipo da AIH: 1 = normal, 5 = continuação de internação longa |
| `MUNIC_RES` | Município de **residência** do paciente (código IBGE de 6 dígitos) |
| `DT_INTERNA` | Data da internação (AAAAMMDD) |
| `DIAG_PRINC` | Diagnóstico principal (CID-10) |
| `IDADE`, `COD_IDADE` | Idade e a unidade dela (2 = dias, 3 = meses, 4 = anos, 5 = 100+ anos) |
| `SEXO` | 1 = masculino, 3 = feminino |
| `MORTE` | 1 se o paciente morreu na internação |
| `DIAS_PERM`, `VAL_TOT` | Dias internado e valor pago (R$) |

In [ ]:
COLUNAS = ["N_AIH", "IDENT", "MUNIC_RES", "DT_INTERNA", "DIAG_PRINC", "IDADE", "COD_IDADE",
           "SEXO", "MORTE", "DIAS_PERM", "VAL_TOT", "ANO_CMPT", "MES_CMPT"]

def ler_sih(arquivo, uf):
    caminho = RAW / arquivo
    disponiveis = pq.read_schema(caminho).names
    faltando = [c for c in COLUNAS if c not in disponiveis]
    if faltando:
        print(f"{uf}: colunas ausentes no arquivo -> {faltando}")
    df = pd.read_parquet(caminho, columns=[c for c in COLUNAS if c in disponiveis])
    df["UF"] = uf
    return df

sih = pd.concat([ler_sih("sih_am_2023_2024.parquet", "AM"),
                 ler_sih("sih_rs_2024.parquet", "RS")], ignore_index=True)

print(f"{len(sih):,} internações x {sih.shape[1]} colunas".replace(",", "."))
display(sih.groupby("UF").size().rename("internações").to_frame())
sih.head()

In [ ]:
# Tipos, faltantes e duplicatas
resumo = pd.DataFrame({
    "tipo": sih.dtypes.astype(str),
    "faltantes": sih.isna().sum(),
    "faltantes_%": (sih.isna().mean() * 100).round(2),
    "valores_distintos": sih.nunique(),
})
display(resumo)

print("Linhas duplicadas por completo:", sih.duplicated().sum())
if "N_AIH" in sih:
    print("N_AIH repetido:", sih.duplicated(subset=["N_AIH"]).sum())
if "IDENT" in sih:
    print("\nTipo de AIH (IDENT):")
    print(sih["IDENT"].value_counts(dropna=False))

## 4. Tratamento

Decisões, cada uma com o motivo:

1. **Só AIH do tipo 1.** A AIH tipo 5 é a continuação de uma internação longa do **mesmo** paciente; contar as duas duplicaria a internação.
2. **Duplicatas de `N_AIH`:** mantemos a primeira ocorrência (o mesmo número não pode ser duas internações).
3. **Mês pela data da internação, não pelo mês de processamento.** Os arquivos do SIH são organizados pelo mês em que o hospital *cobrou* (competência), que pode vir meses depois. O que importa para a história é quando a pessoa adoeceu.
4. **Janela de análise:** AM de jan/2023 a dez/2024 e RS de jan/2024 a dez/2024. Internações anteriores que só foram cobradas dentro do período ficam de fora.
5. **Só residentes do próprio estado.** Usamos o município de residência, porque é lá que a pessoa viveu a seca ou a enchente.
6. **Idade em anos**, convertendo dias e meses para 0 e somando 100 quando `COD_IDADE` = 5.

In [ ]:
n0 = len(sih)
log = []

if "IDENT" in sih:
    sih = sih[sih["IDENT"].astype(str).str.strip() == "1"]
    log.append(("Remove AIH de continuação (IDENT != 1)", n0 - len(sih)))

n1 = len(sih)
if "N_AIH" in sih:
    sih = sih.drop_duplicates(subset=["N_AIH"], keep="first")
    log.append(("Remove N_AIH repetido", n1 - len(sih)))

# data da internação -> mês
sih["data_internacao"] = pd.to_datetime(sih["DT_INTERNA"].astype(str), format="%Y%m%d", errors="coerce")
sih["mes"] = sih["data_internacao"].dt.to_period("M").dt.to_timestamp()

n2 = len(sih)
janela = (((sih["UF"] == "AM") & sih["mes"].between("2023-01-01", "2024-12-01")) |
          ((sih["UF"] == "RS") & sih["mes"].between("2024-01-01", "2024-12-01")))
sih = sih[janela]
log.append(("Fora da janela de análise (ou data inválida)", n2 - len(sih)))

# residentes do próprio estado (2 primeiros dígitos do código IBGE)
COD_UF = {"AM": "13", "RS": "43"}
sih["MUNIC_RES"] = sih["MUNIC_RES"].astype(str).str.strip().str[:6]
n3 = len(sih)
sih = sih[sih["MUNIC_RES"].str[:2] == sih["UF"].map(COD_UF)]
log.append(("Residente de outro estado", n3 - len(sih)))

# idade em anos
idade = pd.to_numeric(sih["IDADE"], errors="coerce")
unidade = sih["COD_IDADE"].astype(str).str.strip()
sih["idade_anos"] = np.select([unidade.isin(["2", "3"]), unidade == "5"], [0, idade + 100], default=idade)
sih["faixa_etaria"] = pd.cut(sih["idade_anos"], bins=[-1, 4, 14, 59, 200],
                             labels=["0–4", "5–14", "15–59", "60+"])

display(pd.DataFrame(log, columns=["passo", "linhas removidas"]))
print(f"Restam {len(sih):,} internações".replace(",", "."))

### Classificando os diagnósticos

Pegamos as 3 primeiras posições do CID-10 do diagnóstico principal e marcamos o grupo de doença.

In [ ]:
cid3 = sih["DIAG_PRINC"].astype(str).str.strip().str.upper().str[:3]

FECO_ORAL = [f"A0{i}" for i in range(10)] + ["B15"]          # diarreias, infecções intestinais, hepatite A
LEPTO = ["A27"]
DENGUE = ["A90", "A91", "A97"]
RESPIRATORIAS = [f"J{i:02d}" for i in range(0, 23)]          # J00–J22

sih["grupo"] = np.select(
    [cid3.isin(FECO_ORAL), cid3.isin(LEPTO), cid3.isin(DENGUE), cid3.isin(RESPIRATORIAS)],
    ["Diarreias e hepatite A", "Leptospirose", "Dengue", "Respiratórias"],
    default="Outras")
sih["drsai"] = sih["grupo"].isin(["Diarreias e hepatite A", "Leptospirose", "Dengue"])

tabela = (sih.groupby(["UF", "grupo"]).size().unstack(0).fillna(0).astype(int))
tabela.loc["Total DRSAI"] = sih[sih["drsai"]].groupby("UF").size()
display(tabela)

### Agregando por município e mês

A base de modelagem terá **uma linha por município por mês**. Montamos a grade completa (todo município em todo mês, com zero onde não houve internação) e juntamos população e PIB per capita da base consolidada da fase 1 para calcular taxas por 100 mil habitantes.

In [ ]:
# perfil dos municípios (fase 1): população e PIB per capita
perfil = pd.read_csv(PROCESSED / "base_consolidada_municipios.csv", encoding="utf-8-sig")
perfil = perfil[perfil["uf"].isin(["AM", "RS"])]
perfil["MUNIC_RES"] = perfil["codigo_ibge_6"].astype(str)
perfil = perfil[["MUNIC_RES", "nome_municipio", "uf", "populacao", "pib_per_capita_calculado"]]
print(perfil.groupby("uf").size().rename("municípios"))

# contagens por município e mês
d = sih.assign(
    drsai_0a4=sih["drsai"] & (sih["faixa_etaria"] == "0–4"),
    drsai_60mais=sih["drsai"] & (sih["faixa_etaria"] == "60+"),
)
agregado = d.groupby(["MUNIC_RES", "mes"]).agg(
    internacoes_total=("grupo", "size"),
    drsai=("drsai", "sum"),
    drsai_0a4=("drsai_0a4", "sum"),
    drsai_60mais=("drsai_60mais", "sum"),
)
por_grupo = d.pivot_table(index=["MUNIC_RES", "mes"], columns="grupo", aggfunc="size", fill_value=0)
agregado = agregado.join(por_grupo).reset_index()

# grade completa município x mês
meses = {"AM": pd.date_range("2023-01-01", "2024-12-01", freq="MS"),
         "RS": pd.date_range("2024-01-01", "2024-12-01", freq="MS")}
grade = pd.concat([pd.MultiIndex.from_product([perfil.loc[perfil["uf"] == uf, "MUNIC_RES"], m],
                                              names=["MUNIC_RES", "mes"]).to_frame(index=False).assign(uf=uf)
                   for uf, m in meses.items()])
base = (grade.merge(perfil, on=["MUNIC_RES", "uf"], how="left")
             .merge(agregado, on=["MUNIC_RES", "mes"], how="left"))
contagens = [c for c in base.columns if c not in ["MUNIC_RES", "mes", "uf", "nome_municipio", "populacao", "pib_per_capita_calculado"]]
base[contagens] = base[contagens].fillna(0).astype(int)
base["taxa_drsai_100k"] = base["drsai"] / base["populacao"] * 1e5

print(f"Base município x mês: {base.shape[0]:,} linhas x {base.shape[1]} colunas".replace(",", "."))
print("Internações de residentes sem município na base IBGE:",
      int(agregado.loc[~agregado["MUNIC_RES"].isin(perfil["MUNIC_RES"]), "internacoes_total"].sum()))
base.head()

## 5. Análise univariada

### Quantas internações por mês? (checagem de completude)

Antes de qualquer conclusão, conferimos se os últimos meses estão completos. Como a cobrança pode chegar meses depois, internações de fim de 2024 podem estar em arquivos de 2025, que **não** foram coletados. Se as barras do fim do período caírem de repente, é falta de registro, não queda de doença.

In [ ]:
mensal = base.groupby(["uf", "mes"])["internacoes_total"].sum().reset_index()
fig, ax = plt.subplots(figsize=(10, 4))
for uf, g in mensal.groupby("uf"):
    ax.plot(g["mes"], g["internacoes_total"] / 1000, marker="o", color=CORES_UF[uf], label=uf)
ax.set_title("Internações no SUS por mês (todas as causas)")
ax.set_ylabel("Internações (mil)")
ax.set_xlabel("Mês da internação")
ax.legend(title="Estado")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b/%y"))
plt.tight_layout(); plt.show()

**O que o gráfico mostra:** _a preencher depois de executar (uma frase com o achado, não a descrição do gráfico)._

### [VÍDEO] Internações por doenças ligadas à água ao longo do tempo

Taxa mensal de internações por DRSAI a cada 100 mil habitantes em cada estado. As faixas marcam os eventos (**datas aproximadas**, a serem substituídas pelas da Defesa Civil): seca no AM em set–dez/2023 e set–nov/2024; enchente no RS em mai/2024.

In [ ]:
EVENTOS = [("AM", "2023-09-01", "2023-12-31", "seca"),
           ("AM", "2024-09-01", "2024-11-30", "seca"),
           ("RS", "2024-04-28", "2024-05-31", "enchente")]

pop_uf = perfil.groupby("uf")["populacao"].sum()
serie = base.groupby(["uf", "mes"])["drsai"].sum().reset_index()
serie["taxa"] = serie["drsai"] / serie["uf"].map(pop_uf) * 1e5

fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True, gridspec_kw={"width_ratios": [2, 1]})
for ax, uf in zip(axs, ["AM", "RS"]):
    g = serie[serie["uf"] == uf]
    ax.plot(g["mes"], g["taxa"], marker="o", color=CORES_UF[uf])
    for (u, ini, fim, rotulo) in EVENTOS:
        if u == uf:
            ax.axvspan(pd.Timestamp(ini), pd.Timestamp(fim), color=CORES_UF[uf], alpha=0.15)
            ax.annotate(rotulo, xy=(pd.Timestamp(ini), 1), xycoords=("data", "axes fraction"),
                        xytext=(3, -12), textcoords="offset points", fontsize=9, color=CORES_UF[uf])
    ax.set_title({"AM": "Amazonas (seca)", "RS": "Rio Grande do Sul (enchente)"}[uf])
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b/%y"))
    ax.tick_params(axis="x", rotation=45)
axs[0].set_ylabel("Internações por DRSAI\npor 100 mil habitantes")
fig.suptitle("Internações por doenças ligadas à água, por mês", fontweight="bold")
plt.tight_layout(); plt.show()

**O que o gráfico mostra:** _a preencher depois de executar (uma frase com o achado, não a descrição do gráfico)._

### Quem rompe primeiro: cada doença no seu tempo

Mesma série, separada por grupo de doença. Cada painel tem a sua escala, porque as respiratórias são muito mais numerosas; o que interessa é **o formato** da curva e **quando** ela sobe.

In [ ]:
grupos = ["Diarreias e hepatite A", "Leptospirose", "Dengue", "Respiratórias"]
longo = (base.groupby(["uf", "mes"])[grupos].sum().reset_index()
             .melt(id_vars=["uf", "mes"], var_name="grupo", value_name="internacoes"))
longo["taxa"] = longo["internacoes"] / longo["uf"].map(pop_uf) * 1e5

g = sns.relplot(data=longo, x="mes", y="taxa", hue="uf", palette=CORES_UF, col="grupo", col_wrap=2,
                kind="line", marker="o", height=3, aspect=1.6, facet_kws={"sharey": False})
for ax in g.axes.flat:
    for (u, ini, fim, _) in EVENTOS:
        ax.axvspan(pd.Timestamp(ini), pd.Timestamp(fim), color=CORES_UF[u], alpha=0.10)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b/%y"))
    ax.tick_params(axis="x", rotation=45)
g.set_axis_labels("Mês da internação", "Por 100 mil hab.")
g.set_titles("{col_name}")
g.fig.suptitle("Internações por grupo de doença e por mês", y=1.02, fontweight="bold")
plt.show()

**O que o gráfico mostra:** _a preencher depois de executar (uma frase com o achado, não a descrição do gráfico)._

### Idade de quem é internado

Comparamos a distribuição por faixa etária das internações por DRSAI com a de todas as internações. Se 0–4 anos e 60+ pesarem mais nas DRSAI, a hipótese 2 ganha força.

In [ ]:
comp = (sih.assign(tipo=np.where(sih["drsai"], "DRSAI", "Todas as outras"))
           .groupby(["UF", "tipo"])["faixa_etaria"].value_counts(normalize=True)
           .rename("proporcao").reset_index())
comp["proporcao"] *= 100

g = sns.catplot(data=comp, x="faixa_etaria", y="proporcao", hue="tipo", col="UF", kind="bar",
                palette=["#444444", "#BBBBBB"], height=3.6, aspect=1.3)
g.set_axis_labels("Faixa etária (anos)", "% das internações")
g.set_titles("{col_name}")
g.fig.suptitle("Distribuição por idade: DRSAI × demais internações", y=1.03, fontweight="bold")
plt.show()

**O que o gráfico mostra:** _a preencher depois de executar (uma frase com o achado, não a descrição do gráfico)._

## 6. Análise bivariada (primeira olhada)

### [VÍDEO] Município pobre adoece mais?

Dividimos os municípios de cada estado em três grupos de PIB per capita (tercis: mais pobres, intermediários e mais ricos) e comparamos a taxa anual de internações por DRSAI. Cada "violino" mostra como os municípios do grupo se espalham; a linha do meio é a mediana.

**Cuidado (falácia ecológica):** isto compara **municípios**, não pessoas. Se os municípios pobres têm taxa maior, não dá para concluir que, dentro de cada município, os mais pobres adoecem mais.

In [ ]:
anual = (base.assign(ano=base["mes"].dt.year)
             .groupby(["uf", "ano", "MUNIC_RES", "nome_municipio", "populacao", "pib_per_capita_calculado"], as_index=False)
             ["drsai"].sum())
anual["taxa_drsai_100k"] = anual["drsai"] / anual["populacao"] * 1e5
anual["periodo"] = anual["uf"] + " " + anual["ano"].astype(str)
anual["renda"] = (anual.groupby("periodo")["pib_per_capita_calculado"]
                       .transform(lambda s: pd.qcut(s, 3, labels=["Mais pobres", "Intermediários", "Mais ricos"])))

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.violinplot(data=anual, x="periodo", y="taxa_drsai_100k", hue="renda", cut=0, inner="quartile",
               palette=["#8C2D04", "#E6A157", "#F3E1C7"], ax=ax)
ax.set_title("Taxa anual de internações por DRSAI por município, segundo o PIB per capita")
ax.set_xlabel("")
ax.set_ylabel("Internações por DRSAI\npor 100 mil hab. no ano")
ax.legend(title="PIB per capita (tercil)")
plt.tight_layout(); plt.show()

display(anual.groupby(["periodo", "renda"], observed=True)["taxa_drsai_100k"].describe()[["count", "50%", "mean", "max"]].round(1))

**O que o gráfico mostra:** _a preencher depois de executar (uma frase com o achado, não a descrição do gráfico)._

## 7. Salvar a base intermediária

A base município × mês vai para `data/processed/`. Quando as ampliações chegarem, ela ganha o tipo de evento, os meses desde o início do evento e o "normal" de cada município, e daí saem os alvos.

In [ ]:
saida = PROCESSED / "drsai_municipio_mes.csv"
base.to_csv(saida, index=False, encoding="utf-8")
print("Salvo:", saida, base.shape)

## Achados preliminares

_A preencher depois de executar: o que confirma ou contradiz cada hipótese da seção 1._

## Próximos passos

- Juntar as Ampliações 1 e 2 (datas oficiais dos eventos e anos de referência).
- Definir o "normal" de cada município e construir os alvos `excesso_drsai` e `excesso_taxa_100k`.
- Escrever o achado de tensão (o "MAS" da história) e qual gráfico o sustenta.